In [ ]:
# import basic stuff
import sklearn
import pandas as pd
import numpy as np
import itertools
from tqdm import tqdm

# import tensorflow and keras stuff
!pip install tensorflow_probability
import tensorflow_probability as tfp
import tensorflow as tf

tfd = tfp.distributions
from keras.layers import *
from keras.models import *
from keras.callbacks import *
from keras.optimizers import *
from keras.losses import *
from keras.regularizers import *
import keras.backend as K

# import kflod stuff
from sklearn.model_selection import KFold

# import preprocessing functions
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

# import comparison models
from xgboost import XGBClassifier, XGBRegressor


!pip install interpret
from interpret.glassbox import (
    ExplainableBoostingClassifier,
    ExplainableBoostingRegressor,
)



# plotting
import matplotlib.pyplot as plt


from scipy.stats import entropy, wasserstein_distance

!pip install properscoring
import properscoring as ps


In [ ]:
!pip install nodegam
from nodegam.sklearn import NodeGAMRegressor, NodeGAMClassifier
from nodegam.gams.MySpline import MySplineLogisticGAM, MySplineGAM
from nodegam.gams.MyEBM import MyExplainableBoostingClassifier, MyExplainableBoostingRegressor
from nodegam.gams.MyXGB import MyXGBOnehotClassifier, MyXGBOnehotRegressor
from nodegam.gams.MyBagging import MyBaggingClassifier, MyBaggingRegressor
from nodegam.utils import sigmoid_np, average_GAM_dfs
from nodegam.vis_utils import vis_GAM_effects

In [ ]:
from sklearn.metrics import mean_gamma_deviance
def gamma_dev(targets, predictions):
    return mean_gamma_deviance(targets, predictions)

def kl_divergence(predicted_mu, predicted_sigma, y_test):
    y_dist = np.expand_dims(y_test, 1)
    y_dist = y_dist.astype(np.float32)


    if predicted_mu.shape != (len(y_test), 1):
        predicted_mu = np.expand_dims(y_test, 1)
    try:
        predicted_mu = predicted_mu.astype(np.float32)
    except:
        pass

    try:
        predicted_sigma = predicted_sigma.astype(np.float32)
    except:
        pass

    t = tfd.ExpInverseGamma(concentration=y_dist, scale=np.std(y_dist))

    p = tfd.ExpInverseGamma(concentration=predicted_mu, scale=predicted_sigma)

    kl = tf.reduce_mean(np.log(tfd.kl_divergence(t, p, allow_nan_stats=True)))

    return kl.numpy()




def compute_wasserstein_distance(p, q):
    return wasserstein_distance(p, q)


In [ ]:

####################################### NAM EXU-activation Layer
class ExuLayer(tf.keras.layers.Layer):
    def __init__(self, units=32, input_dim=32):
        super(ExuLayer, self).__init__()
        w_init = tf.random_normal_initializer()
        self.w = tf.Variable(
            initial_value=w_init(shape=(input_dim, units), dtype="float32"),
            trainable=True,
        )
        b_init = tf.zeros_initializer()
        self.b = tf.Variable(
            initial_value=b_init(shape=(units,), dtype="float32"), trainable=True
        )

    def call(self, inputs):
        return tf.clip_by_value(tf.matmul(inputs, tf.exp(self.w)) + self.b, 0, 1)

In [ ]:
class CustomPipeline(Pipeline):
    """Custom sklearn Pipeline to transform data."""

    def apply_transformation(self, x):
        """Applies all transforms to the data, without applying last estimator.

        Args:
          x: Iterable data to predict on. Must fulfill input requirements of first
            step of the pipeline.

        Returns:
          xt: Transformed data.
        """
        xt = x
        for _, transform in self.steps[:-1]:
            xt = transform.fit_transform(xt)
        return xt


def transform_data(df):
    """Apply a fixed set of transformations to the pd.Dataframe `df`.

    Args:
      df: Input dataframe containing features.

    Returns:
      Transformed dataframe and corresponding column names. The transformations
      include (1) encoding categorical features as a one-hot numeric array, (2)
      identity `FunctionTransformer` for numerical variables. This is followed by
      scaling all features to the range (-1, 1) using min-max scaling.
    """
    column_names = df.columns
    new_column_names = []
    is_categorical = np.array([dt.kind == "O" for dt in df.dtypes])
    categorical_cols = df.columns.values[is_categorical]
    numerical_cols = df.columns.values[~is_categorical]
    for index, is_cat in enumerate(is_categorical):
        col_name = column_names[index]
        if is_cat:
            new_column_names += [
                "{}: {}".format(col_name, val) for val in set(df[col_name])
            ]
        else:
            new_column_names.append(col_name)
    cat_ohe_step = ("ohe", OneHotEncoder(sparse=False, handle_unknown="ignore"))

    cat_pipe = Pipeline([cat_ohe_step])
    num_pipe = Pipeline([("identity", FunctionTransformer(validate=True))])
    transformers = [
        ("cat", cat_pipe, categorical_cols),
        ("num", num_pipe, numerical_cols),
    ]
    column_transform = ColumnTransformer(transformers=transformers)

    pipe = CustomPipeline(
        [
            ("column_transform", column_transform),
            ("min_max", MinMaxScaler((-1, 1))),
            ("dummy", None),
        ]
    )
    df = pipe.apply_transformation(df)
    return df, new_column_names


In [ ]:



######################################################### Model builder


############################### Helper functions for building MLP, NAM and NAMLSS
def built_DNN(input, output_activation="linear", output_num=1):
    x = Dense(512, "relu")(input)
    x = Dropout(0.5)(x)
    x = Dense(256, "relu")(x)
    x = Dense(50, "relu")(x)
    x = Dense(output_num, activation=output_activation, use_bias=False)(x)
    model_dnn = Model(inputs=input, outputs=x)
    model_dnn.reset_states()
    return model_dnn


def LINEAR(x):
    return x


def gamma_activation(x):
    value = K.maximum(1/tf.math.softplus(x),tf.math.softplus(x))
    return value


################# MLP
def MLP(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    output_activation="softplus",
):
    inps = Input(shape=(features_train.shape[1],))
    model = built_DNN(inps, output_activation=output_activation)

    model.compile(
        loss=POINT_LOSS, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=features_train,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    loc_pred = model.predict(features_test)
    loc_pred = np.array([loc_pred[i][0] for i in range(len(loc_pred))], dtype=np.float64)
    likelihood = LL_EVAL(loc_pred, labels_test)

    ll, point_loss, kl, ws = LL_EVAL(loc_pred, labels_test)

    return ll, point_loss, kl, ws


######################## Distributional DNN


def DDNN(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    distribution=tfd.InverseGamma,
    loc_activation=gamma_activation,
    scale_activation=tf.math.softplus,
    output_num=2,
):
    # Create inputs

    def keras_model(input):
        x = Dense(512, "relu")(input)
        x = Dropout(0.5)(x)
        x = Dense(256, "relu")(x)
        x = Dense(50, "relu")(x)
        alpha = Dense(1, activation=gamma_activation, use_bias=False)(x)
        beta = Dense(1, activation=tf.math.softplus, use_bias=False)(x)

        x = concatenate([alpha, beta])
        model_dnn = Model(inputs=input, outputs=x)
        model_dnn.reset_states()
        return model_dnn

    inps = Input(shape=(features_train.shape[1],))
    ms = keras_model(inps)
    z = ms.output

    # built distributional layer
    # Change for when dist params have different names
    p_y = tfp.layers.DistributionLambda(
        lambda x: distribution(
            concentration=x[:, 0], scale=x[:, 1]
        )
    )(z)

    model = Model(inputs=ms.input, outputs=p_y)

    def NLL(y_true, y_hat):
        return -y_hat.log_prob(y_true)

    model.compile(
        loss=NLL, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=features_train,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    # Evaluate model
    preds = ms(features_test)
    mu_preds = np.array(preds[:, 0])
    sigma_preds = np.array(preds[:, 1])


    ll, point_loss, kl, ws = LL_EVAL(mu_preds, labels_test, sigma_preds)

    return ll, point_loss, kl, ws


######################################### NAM


def NAM(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    output_activation="softplus",
):
    inps = [Input(shape=(1,)) for _ in range(features_train.shape[1])]

    # define submodels
    # same architecture as for DNN and MLP
    ms = [
        built_DNN(inps[i], output_activation=output_activation)
        for i in range(features_train.shape[1])
    ]
    z = sum([m.output for m in ms])
    model = Model(inputs=[m.input for m in ms], outputs=z)

    model.compile(
        loss=POINT_LOSS, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    training_data = [features_train[:,i] for i in range(features_train.shape[1])]
    eval_data = [features_test[:,i] for i in range(features_test.shape[1])]

    history = model.fit(
        x=training_data,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    loc_pred = model.predict(eval_data)
    loc_pred = np.array([loc_pred[i][0] for i in range(len(loc_pred))], dtype=np.float64)
    ll, point_loss, kl, ws = LL_EVAL(loc_pred, labels_test)

    return ll, point_loss, kl, ws


############################################ NAMLSS


def define_models_scale(input):
    x = Dense(512, "relu")(input)
    x = Dropout(0.5)(x)
    x = Dense(256, "relu")(x)
    x = Dense(50, "relu")(x)
    x = Dense(1, activation="linear", use_bias=False)(x)
    x = Model(inputs=input, outputs=x)
    #x.reset_states()
    return x


def NAMLSS(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    distribution=tfd.InverseGamma,
    loc_activation=gamma_activation,
    scale_activation=tf.math.softplus,
    output_num=1,
):



    training_data = 2 * [features_train[:, i] for i in range(features_train.shape[1])]
    eval_data = 2 * [features_test[:, i] for i in range(features_test.shape[1])]

    inps = [Input(shape=(1,)) for _ in range(2 * features_train.shape[1])]

    ms = [built_DNN(inps[i]) for i in range(features_train.shape[1])]
    ms += [
        define_models_scale(inps[i + features_train.shape[1]])
        for i in range(features_train.shape[1])
    ]

    z1 = loc_activation(sum([m.output for m in ms[: features_train.shape[1]]]))
    z2 = scale_activation(sum([m.output for m in ms[features_train.shape[1] :]]))

    z = concatenate([z1, z2])

    # Change for when dist params have different names
    p_y = tfp.layers.DistributionLambda(
        lambda x: distribution(
            concentration=x[:, 0], scale=x[:, 1]
        )
    )(z)

    model = Model(inputs=[m.input for m in ms], outputs=p_y)

    def NLL(y_true, y_hat):
        return -y_hat.log_prob(y_true)

    model.compile(
        loss=NLL, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=training_data,
        y=labels_train,
        validation_split=0.2,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    preds = [ms[idx].predict(eval_data[idx], verbose=0) for idx in range(len(eval_data))]
    preds_mu = sum(preds[:features_test.shape[1]])
    preds_sigma = sum(preds[features_test.shape[1]:])

    mu_preds = loc_activation(preds_mu)
    sigma_preds = scale_activation(preds_sigma)



    mu = np.array([mu_preds[i][0] for i in range(len(mu_preds))])
    sigma = np.array([sigma_preds[i][0] for i in range(len(sigma_preds))])



    ll, point_loss, kl, ws = LL_EVAL(mu, labels_test, sigma)

    return ll, point_loss, kl, ws


#### NAMLSS 2
def NA2MLSS(
    features_train,
    labels_train,
    features_test,
    labels_test,
    metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"],
    distribution=tfd.InverseGamma,
    loc_activation=gamma_activation,
    scale_activation=tf.math.softplus,
    output_num=2,
):
    training_data = [features_train[:, i] for i in range(features_train.shape[1])]
    eval_data = [features_test[:, i] for i in range(features_test.shape[1])]

    inps = [Input(shape=(1,)) for _ in range(features_train.shape[1])]

    ms = [
        built_DNN(inps[i], output_num=output_num)
        for i in range(features_train.shape[1])
    ]

    z = sum([m.output for m in ms])

    # Change for when dist params have different names
    p_y = tfp.layers.DistributionLambda(
        lambda x: distribution(
            concentration=loc_activation(x[:, 0]), scale=scale_activation(x[:, 1])
        )
    )(z)

    model = Model(inputs=[m.input for m in ms], outputs=p_y)

    def NLL(y_true, y_hat):
        return -y_hat.log_prob(y_true)

    model.compile(
        loss=NLL, metrics=metrics, optimizer=Adam(learning_rate=LEARNING_RATE)
    )

    history = model.fit(
        x=training_data,
        y=labels_train,
        epochs=NUM_EPOCHS,
        callbacks=[EARLY_STOPPING, REDUCE_LR],
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    preds = [ms[idx].predict(eval_data[idx], verbose=0) for idx in range(len(eval_data))]

    preds = sum(preds)
    mu, sigma = preds[:,0], preds[:, 1]
    mu_preds = loc_activation(mu)
    sigma_preds = scale_activation(sigma)

    ll, point_loss, kl, ws = LL_EVAL(mu_preds, labels_test, sigma_preds)

    return ll, point_loss, kl, ws


###################### XGBoost
def XGB(features_train, labels_train, features_test, labels_test, regression=True):
    if regression:
        model = XGBRegressor()
    else:
        model = XGBClassifier()
    model.fit(features_train, np.log(labels_train))

    preds = np.exp(model.predict(features_test))

    ll, point_loss, kl, ws= LL_EVAL(np.float64(preds), labels_test)

    return ll, point_loss, kl, ws




############## EBM
def EBM(features_train, labels_train, features_test, labels_test, regression=True):
    if regression:
        model = ExplainableBoostingRegressor()
    else:
        model = ExplainableBoostingClassifier()

    model.fit(features_train, np.log(labels_train))

    preds = np.exp(model.predict(features_test))

    ll, point_loss, kl,  ws = LL_EVAL(np.float64(preds), labels_test)

    return ll, point_loss, kl,  ws


############################## NODEGAM
def NODEGAM(features_train, labels_train, features_test, labels_test, regression=True):
    if regression:
        model = NodeGAMRegressor(
            in_features=features_train.shape[1], verbose=0, seed=141, ga2m=0
        )
    else:
        model = NodeGAMClassifier(
            in_features=features_train.shape[1],
            verbose=0,
            seed=141,
            ga2m=0,
        )

    X_train = pd.DataFrame(np.vstack([features_train])).reset_index(drop=True)
    X_test = pd.DataFrame(np.vstack([features_test])).reset_index(drop=True)
    record = model.fit(X_train, np.log(np.array(labels_train)))
    preds = np.exp(model.predict(X_test))

    ll, point_loss, kl,  ws = LL_EVAL(np.float64(preds), labels_test)

    return ll, point_loss, kl,  ws


In [ ]:

if __name__ == "__main__":
    # task:
    REGRESSION = True
    # general arguments
    BATCH_SIZE = 1024
    NUM_EPOCHS = 2000
    LEARNING_RATE = 0.001
    NUM_FOLDS = 5

    # loss func for point estimators
    POINT_LOSS = "mse"

    EARLY_STOPPING = EarlyStopping(
        patience=100, restore_best_weights=True, min_delta=1e-05, monitor="loss"
    )

    METRICS = [tf.keras.metrics.RootMeanSquaredError(name="rmse"), "mse"]

    REDUCE_LR = ReduceLROnPlateau(
        monitor="loss", factor=0.95, patience=25, min_delta=1e-05
    )

    KFOLD = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=101)

    DISTRIBUTION = tfd.InverseGamma

    # Define distribution that is modelled
    def LL_EVAL(loc, y_true, scale=None):
        if scale is None:

          concentration = loc**2 / np.var(loc) + 2    # alpha
          scale = loc * ( (loc**2) / np.var(loc) + 1 )    # beta
          dist = DISTRIBUTION(concentration, scale=scale)
        else:
          dist = DISTRIBUTION(concentration=loc, scale=scale)
        ll = - tf.reduce_sum(dist.log_prob(value=y_true)).numpy()

        if scale is None:
          point_loss = gamma_dev(y_true, loc)
        else:

          real_preds = scale / (loc - 1)
          point_loss = gamma_dev(y_true, real_preds)

        wasserstein = compute_wasserstein_distance(y_true, loc)
        if scale is None:
          kl_div = kl_divergence(loc, np.std(loc), y_true)

        else:
          kl_div = kl_divergence(loc, scale, y_true)


        print(f"LL: {ll}, Gamma Deviance: {point_loss}, KL_Div: {kl_div},  Wasserstein Distance: {wasserstein}")

        return ll, point_loss, kl_div, wasserstein


    from scipy import stats
    df = pd.read_csv("/content/melbourne.csv")
    X = df.drop(["name", "license", "host_name", "host_id", "neighbourhood", "neighbourhood_group", "last_review", "availability_365", "id"], axis=1)

    X = X.reset_index(drop=True)
    X["price"] = X["price"].astype(float)
    X = X.dropna()
    X = X.reset_index(drop=True)
    targets = X.pop("price")

    # Always use transform_data function on X
    features, cols = transform_data(X)

    fold_no = 1

    model_list = [
        "NAMLSS",
        "NA2MLSS",
        "NAM",
        "XGBOOST",
        "EBM",
        "NODEGAM",
        "MLP",
        "DDNN",
    ]

    results = pd.DataFrame(columns=["Model", "Likelihood", "GAMMA_DEV", "KL",  "Wasserstein"])

    for mod in model_list:
        print(mod)
        ll_per_fold = []
        gamma_dev_per_fold = []
        kl_div_per_fold = []
        ws_dis_per_fold = []


        for train, test in KFOLD.split(features, targets):
            if mod == "DDNN":
                ll, pl, kl,  ws= DDNN(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    distribution=DISTRIBUTION,
                    loc_activation=gamma_activation,
                    scale_activation=tf.math.softplus,
                    output_num=2,
                )
            elif mod == "MLP":
                ll, pl, kl,  ws= MLP(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    output_activation="linear",
                )

            elif mod == "NAM":
                ll, pl, kl,  ws = NAM(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    output_activation="linear",
                )
            elif mod == "XGBOOST":
                ll, pl, kl,  ws= XGB(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    REGRESSION,
                )
            elif mod == "EBM":
                ll, pl, kl, ws = EBM(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    REGRESSION,
                )
            elif mod == "NODEGAM":
                ll, pl, kl, ws= NODEGAM(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    REGRESSION,
                )
            elif mod == "NAMLSS":
                ll, pl, kl, ws = NAMLSS(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    distribution=DISTRIBUTION,
                    loc_activation=gamma_activation,
                    scale_activation=tf.math.softplus,
                    output_num=1,
                )
            elif mod == "NA2MLSS":
                ll, pl, kl, ws = NA2MLSS(
                    features[train],
                    targets[train],
                    features[test],
                    targets[test],
                    metrics=METRICS,
                    distribution=DISTRIBUTION,
                    loc_activation=gamma_activation,
                    scale_activation=tf.math.softplus,
                    output_num=2,
                )

            ll_per_fold.append(ll)
            gamma_dev_per_fold.append(pl)
            kl_div_per_fold.append(kl)
            ws_dis_per_fold.append(ws)


        name = NUM_FOLDS*[mod]
        temp_df = pd.DataFrame(np.array((name, ll_per_fold, gamma_dev_per_fold, kl_div_per_fold, ws_dis_per_fold)).T, columns=["Model", "Likelihood", "GAMMA_DEV", "KL", "Wasserstein"])
        results = pd.concat([results, temp_df])


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


NAMLSS


0it [00:00, ?it/s]

0.17267986
1.1420877


1it [00:48, 48.60s/it]

...

5470.459610532364 103.44894682928756 3.5673404 52.19763630241124
NODEGAM


0it [00:00, ?it/s]

Normalize y. mean = 4.922407916235862, std = 0.7792036851279125


0it [00:00, ?it/s]


RuntimeError: ignored

In [ ]:
results = results.astype({"Likelihood": float})
results = results.astype({"GAMMA_DEV": float})
results = results.astype({"CRSP": float})
results = results.astype({"Wasserstein": float})
results = results.astype({"KL": float})
results = results.astype({"SKL": float})
results.groupby("Model").mean()

In [ ]:
results.groupby("Model").std()